# Modeling Human Activity States Using Hidden Markov Models

## Background and Motivation

Human Activity Recognition (HAR) is a critical component in modern applications such as wearable health monitors, elderly fall detection systems, fitness trackers, and smart home automation. In these systems, continuous data streams from inertial sensors (accelerometers and gyroscopes) embedded in smartphones provide valuable information about a person's physical state. However, the true activity (e.g., walking or standing) is hidden behind noisy sensor measurements, making it a natural candidate for probabilistic sequence modeling.

In this project, we use a Hidden Markov Model (HMM) to infer four human activities (Standing, Walking, Jumping, and Still) from smartphone sensor data collected using the Sensor Logger app. Our unique use case focuses on recognizing activities from two different Android devices with the same sampling rate, demonstrating cross-device generalization for practical HAR deployment.

## 1. Data Loading

We load all 57 cleaned CSV files from the `data/` directory. The files are organized into four subfolders by activity (Standing, Walking, Jumping, Still). Each CSV contains 7 columns: `seconds_elapsed, accel_x, accel_y, accel_z, gyro_x, gyro_y, gyro_z`.

- **Phone 1 (STK-L22):** 28 files, 20ms sampling interval (50 Hz)
- **Phone 2 (Infinix X6855):** 29 files, 20ms sampling interval (50 Hz)

Since both phones share the same 20ms sampling rate, no resampling or harmonization is needed.

In [ ]:
# Import libraries and load all CSV files from data/ directory
# Print summary: total files, per-activity counts, per-phone counts, durations

## 2. Raw Signal Visualization

Before any processing, we visualize the raw accelerometer and gyroscope signals for one sample recording of each activity. This helps us understand the differences in signal patterns:

- **Standing:** Relatively stable accelerometer readings with small fluctuations from body sway; gyroscope near zero
- **Walking:** Rhythmic, periodic oscillations in both accelerometer and gyroscope from the regular step pattern
- **Jumping:** Large, high-amplitude spikes in the accelerometer (especially z-axis) from impact forces
- **Still:** Extremely flat signals with almost no variation, since the phone is stationary on a surface

We plot a 4x2 grid: one row per activity, left column = accelerometer (x, y, z), right column = gyroscope (x, y, z).

In [ ]:
# Plot raw accelerometer and gyroscope signals (4x2 subplot grid)
# One row per activity, left = accelerometer, right = gyroscope

## 3. Windowing

Before extracting features, we divide each continuous recording into fixed-size **windows** (segments). This converts variable-length recordings into uniform-length segments suitable for feature extraction.

**Parameters:**

| Parameter | Value | Justification |
|:---|:---|:---|
| Window Size | 50 samples (1 second at 50 Hz) | Captures a full gait cycle for walking (~0.5-1.0s) and at least one jump cycle. Provides 1 Hz FFT frequency resolution. |
| Step Size | 25 samples (50% overlap) | Increases the number of training samples without losing temporal continuity between consecutive windows. |

**Why 1 second?** At 50 Hz, a 1-second window provides enough samples to capture the periodic patterns in human activities (walking at ~2 Hz, jumping at ~1-2 Hz) while keeping windows short enough that each window contains a single activity.

In [ ]:
# Define windowing function and segment all files into windows
# Print total windows extracted and per-activity counts

## 4. Feature Extraction

We extract both **time-domain** and **frequency-domain** features from each window. These features form the observation vectors for the HMM.

### Time-Domain Features (22 features)

| Feature | Count | Justification |
|:---|:---:|:---|
| Mean (per axis) | 6 | Baseline signal level; Standing has a distinct z-accel value from gravity |
| Standard Deviation (per axis) | 6 | Measures movement intensity; Jumping has high std, Still has near-zero |
| RMS (per axis) | 6 | Overall signal energy; higher for dynamic activities like Walking and Jumping |
| Signal Magnitude Area (SMA) | 1 | Total body acceleration summed across all accelerometer axes |
| Inter-axis Correlation (xy, xz, yz) | 3 | Walking shows correlated axes from the rhythmic, coordinated motion pattern |

### Frequency-Domain Features (6 features)

| Feature | Count | Justification |
|:---|:---:|:---|
| Dominant Frequency (per accel axis) | 3 | Walking peaks at ~2 Hz from regular steps; Still has no dominant peak |
| Spectral Energy (per accel axis) | 3 | Separates high-energy Jumping from low-energy Still |

**Total: 28 features per window.**

The frequency-domain features are computed using the **Fast Fourier Transform (FFT)** on each accelerometer axis.

In [ ]:
# Define feature extraction function
# Extract 28 features per window (22 time-domain + 6 frequency-domain)
# Build feature matrix X_raw and label arrays

### 4.1 Feature Normalization

All 28 features are normalized using **Z-score standardization** (zero mean, unit variance). This is necessary because:

- Features have different units (m/s² for accelerometer, rad/s for gyroscope, Hz for frequency, dimensionless for correlations)
- Gaussian emission distributions in the HMM assume comparable feature scales
- Without normalization, features with larger magnitudes would dominate the distance calculations

Z-score was chosen over min-max normalization because it preserves the shape of the feature distributions and is robust to outliers.

In [ ]:
# Apply Z-score normalization using StandardScaler
# Verify mean ~0 and std ~1 after normalization

## 5. Train / Test Split

We split the data **by file** (not by individual windows) to ensure that the test set contains entirely unseen recordings. This prevents data leakage from overlapping windows of the same recording appearing in both train and test sets.

- **Training set:** All files except the last 2 per activity
- **Test set:** Last 2 files per activity (8 files total)

This gives us held-out recordings for realistic evaluation.

In [ ]:
# Split data by file: hold out last 2 files per activity for testing
# Print train/test sizes and per-activity breakdown

## 6. HMM Model Definition

Our Hidden Markov Model is defined by the following five components:

| Component | Symbol | Description |
|:---|:---:|:---|
| Hidden States | Z | 4 activities: Standing, Walking, Jumping, Still |
| Observations | X | 28-dimensional feature vectors from windowed sensor data |
| Transition Matrix | A (4x4) | Probability of transitioning between activities |
| Emission Parameters | B = {mu_k, sigma_k} | Gaussian mean and diagonal covariance per state |
| Initial Probabilities | pi (4x1) | Probability of starting in each activity |

We use **diagonal covariance** matrices for computational stability and to avoid singularity issues with limited training data. All computations are performed in **log-space** to prevent numerical underflow in the Forward, Backward, and Viterbi algorithms.

The model is implemented from scratch as a `GaussianHMM` class with the following methods:
- `initialize_from_labels()` - set initial parameters from ground-truth labels
- `_forward()` / `_backward()` - Forward and Backward algorithms in log-space
- `fit()` - Baum-Welch (EM) training
- `viterbi()` - Viterbi decoding for most-likely state sequence
- `predict()` - wrapper for Viterbi decoding

In [ ]:
# Define GaussianHMM class with:
# - __init__: initialize parameters (pi, A, means, covars)
# - initialize_from_labels: set params from ground-truth for faster convergence
# - _log_gaussian_pdf: log emission probability under diagonal Gaussian
# - _compute_log_emission: emission matrix for all observations
# - _forward: Forward algorithm in log-space
# - _backward: Backward algorithm in log-space
# - _logsumexp: numerically stable log-sum-exp
# - fit: Baum-Welch (EM) training with convergence check
# - viterbi: Viterbi decoding (dynamic programming in log-space)
# - predict: wrapper for viterbi

## 7. Training with Baum-Welch (EM Algorithm)

We now train the HMM using the **Baum-Welch algorithm**, which is an Expectation-Maximization (EM) procedure for HMMs. It iteratively refines all model parameters to maximize the likelihood of the observed training data.

### Algorithm Steps:

1. **E-Step (Expectation):** Compute posterior state probabilities (gamma) and pairwise transition posteriors (xi) using the Forward-Backward algorithm in log-space
2. **M-Step (Maximization):** Update all parameters (pi, A, means, covariances) to maximize the expected data log-likelihood
3. **Convergence Check:** Stop when |delta log-likelihood| < epsilon = 1e-4

### Initialization:
- Emission means and covariances are initialized from ground-truth labels for faster convergence
- Transition matrix is initialized with 0.95 on the diagonal (high self-transition) and uniform off-diagonal
- Initial state probabilities are set proportional to activity frequencies

First, we build the training sequences (one per file):

In [ ]:
# Build training sequences: one sequence per training file
# Each sequence is a (T_i, 28) array of normalized features

Now we initialize the model and run Baum-Welch training:

In [ ]:
# Initialize HMM with 4 states and 28 features
# Initialize parameters from training labels
# Run Baum-Welch with max_iter=150, tol=1e-4

### Convergence Plot

We plot the total log-likelihood at each iteration to verify that the EM algorithm is converging correctly. A correct implementation should show a monotonically increasing curve that plateaus.

In [ ]:
# Plot log-likelihood vs iteration number

## 8. Transition Matrix Heatmap

The transition matrix **A** captures the probability of moving from one activity state to another between consecutive time windows. We visualize it as a heatmap to understand the learned activity dynamics.

**Expected behavior:**
- High diagonal values (close to 1.0) indicate that activities are persistent - a person who is walking is very likely to still be walking in the next 1-second window
- Small off-diagonal values allow the model to handle genuine activity transitions while resisting spurious state changes from noise

In [ ]:
# Plot transition matrix A as a heatmap using seaborn

## 9. Emission Probability Visualization

Since our HMM uses **Gaussian emissions**, each state k is characterized by a mean vector (mu_k) and a diagonal covariance vector (sigma_k). We visualize these as heatmaps for a subset of key features to understand what signal patterns each activity state has learned.

**Key features to examine:**
- `mean_accel_x/y/z` - baseline acceleration per axis
- `std_accel_x/y/z` - movement intensity
- `sma` - total body acceleration
- `dom_freq_ax` - dominant frequency
- `energy_ax` - spectral energy

We plot two heatmaps side by side: emission means (left) and emission variances (right).

In [ ]:
# Plot emission parameters: means heatmap (left) and variances heatmap (right)
# Use key features subset for readability

## 10. Viterbi Decoding on Training Data

The **Viterbi algorithm** finds the single most likely sequence of hidden states given the observations. It uses dynamic programming in log-space to compute the maximum probability path through the state trellis, then backtracks to recover the optimal state sequence.

We first run Viterbi on the training sequences to verify that the model has learned the correct state assignments. For each activity, we show one training sequence with the true state (black) overlaid with the predicted state (colored).

In [ ]:
# Run Viterbi on training sequences
# Plot true vs predicted state sequences (4x1 subplot grid)

## 11. Evaluation on Unseen Data

We test the trained model on **combined multi-activity sequences** built from held-out test files. Instead of testing each activity file individually (which would be trivially easy), we concatenate the **raw sensor data** from one file per activity in order:

**Standing -> Walking -> Jumping -> Still**

Then we apply windowing to the concatenated signal. This creates a realistic challenge where:
- The model must handle **transitions between activities**
- Windows at activity boundaries contain **mixed signals** from two activities
- The model must correctly decode the full multi-activity sequence

We build 2 such combined test sequences from the 8 held-out files (2 per activity).

### Building the Combined Test Sequences

In [ ]:
# Collect test file raw data grouped by activity
# Build 2 combined sequences by concatenating raw sensor data
# Apply windowing to combined sequences
# Assign ground-truth labels via majority vote per window

### Confusion Matrix

We run Viterbi decoding on both combined test sequences and aggregate the predictions into a confusion matrix. This shows exactly which activities are being confused with each other.

In [ ]:
# Run Viterbi on combined test sequences
# Build and plot confusion matrix heatmap

### Evaluation Metrics

We compute **Sensitivity** (true positive rate), **Specificity** (true negative rate), and **Overall Accuracy** for each activity from the confusion matrix.

In [ ]:
# Compute and print per-activity: Sensitivity, Specificity, Accuracy
# Print overall accuracy

### Decoded Test Sequences Visualization

We visualize the Viterbi-decoded combined test sequences, showing the true activity labels (black) and predicted labels (colored) over time. Dashed vertical lines mark the true activity transition boundaries. Activity labels are annotated above each segment.

In [ ]:
# Plot decoded combined test sequences (2x1 subplot)
# Show true vs predicted, transition boundaries, activity labels

## 12. Analysis and Reflection

### Which activities were easiest/hardest to distinguish?

- **Easiest:** Jumping and Walking. Jumping produces very high accelerometer variance and spectral energy, making it unmistakable. Walking has a characteristic ~2 Hz dominant frequency from the regular step pattern. Both are clearly separable from static activities in the feature space.

- **Hardest:** Standing vs Still. Both are low-movement activities with overlapping feature distributions. The main difference is subtle body sway when standing versus near-zero signal when the phone is placed on a flat surface. This creates ambiguity especially at transition boundaries where windows contain mixed signals.

### How transition probabilities reflect realistic behavior

The learned transition matrix shows high self-transition probabilities (close to 1.0 on the diagonal), which realistically reflects that people tend to continue their current activity rather than rapidly switching every second. This self-transition inertia also acts as a smoothing mechanism, helping the model resist brief spurious state changes caused by noisy sensor observations.

### Effect of sensor noise and sampling rate

Both phones used the same 20ms (50 Hz) sampling rate, simplifying preprocessing since no resampling was needed. However, the two devices (STK-L22 and Infinix X6855) have different sensor hardware, leading to slightly different noise floors and calibration offsets. Z-score normalization helped mitigate these inter-device differences. The 50 Hz rate provides adequate resolution for human activities, which occur at 0.5-5 Hz.

### Potential improvements

1. **More data** from additional participants would improve cross-person generalization
2. **Additional features** such as jerk (derivative of acceleration) and tilt angle could better distinguish Standing from Still
3. **Magnetometer data** could help with orientation-dependent activities
4. **Continuous multi-activity recordings** would provide more realistic transition patterns for training
5. **GMM-HMM** (Gaussian Mixture emissions per state) could capture more complex, multimodal feature distributions